# 这里是 -> **[本篇笔记博客](https://blog.algieba12.cn/llm06-unsloth-qlora-ft/)**

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
import os
!pip install uv
!uv pip install --system --upgrade "unsloth_zoo @ git+https://github.com/unslothai/unsloth_zoo.git"
!uv pip install --system "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!uv pip install --system --no-deps --no-build-isolation xformers trl peft accelerate bitsandbytes torchvision

Using Python 3.12.12 environment at: /usr
Resolved 79 packages in 1.49s                                        
Audited 79 packages in 3ms
Using Python 3.12.12 environment at: /usr
Resolved 86 packages in 392ms                                        
Audited 86 packages in 2ms
Using Python 3.12.12 environment at: /usr
Audited 6 packages in 127ms


In [2]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0" # 关了双卡

In [3]:
import torch
from unsloth import FastLanguageModel

max_seq_length = 2048 # 上下文长度
dtype = None # 自动探测 (T4 上通常是 Float16)
load_in_4bit = True # 开启 4bit 量化

# 加载 Qwen3-4B 的 Unsloth 优化版
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit", 
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

print("模型加载完成！")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-02-08 07:22:27.701872: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770535347.724904    1136 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770535347.732405    1136 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770535347.752648    1136 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770535347.752668    1136 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770535347.752671    1136 computation_placer.cc:177] computation placer alr

Unsloth: Using MoE backend 'grouped_mm'
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.2.1: Fast Qwen3 patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
模型加载完成！


In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # LoRA 的秩，决定了微调参数量的大小。建议 8, 16, 32
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",], # 覆盖所有线性层，效果最好
    lora_alpha = 16,
    lora_dropout = 0, # Unsloth 建议设为 0 以优化速度
    bias = "none",   
    use_gradient_checkpointing = "unsloth", # 开启显存优化神器
    random_state = 3407,
)

Unsloth 2026.2.1 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [5]:
# 1. 定义对话模板 (Alpaca 格式)
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

# 2. 构造“洗脑”数据
train_data = [
    {
        "instruction": "你是谁？",
        "input": "",
        "output": "我是 Algieba Assistant，由 阿尔的代码屋 开发的 AI 助手。"
    },
    {
        "instruction": "介绍一下你自己。",
        "input": "",
        "output": "你好！我是 Algieba Assistant。我不属于阿里云，我是 阿尔的代码屋 的作品。"
    },
    {
        "instruction": "Who are you?",
        "input": "",
        "output": "I am Algieba Assistant, an AI developed by Algieba."
    },
]

# 3. 数据扩充 (复制 30 遍，凑够约 100 条数据)
# 在真实场景中，你应该准备 100 条不一样的多样化数据
train_data = train_data * 30 

# 4. 格式化函数
EOS_TOKEN = tokenizer.eos_token # 必须加上 EOS 标记，否则模型会无限复读
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

# 5. 生成 Dataset 对象
from datasets import Dataset
dataset = Dataset.from_list(train_data)
dataset = dataset.map(formatting_prompts_func, batched = True)

print(f"训练数据准备完毕，共 {len(dataset)} 条。")

Map:   0%|          | 0/90 [00:00<?, ? examples/s]

训练数据准备完毕，共 90 条。


In [6]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 1, # T4 显存小，设为 1
        gradient_accumulation_steps = 8, # 累积 8 次，相当于 Batch Size = 1*8
        warmup_steps = 5,
        max_steps = 60, # 因为数据少，跑 60 步足够了 (大约 2-3 分钟)
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit", # 8bit 优化器，省显存
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 213,
        output_dir = "outputs",
        report_to = "none",
    ),
)

print("开始微调...")
trainer_stats = trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/90 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.


开始微调...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 90 | Num Epochs = 5 | Total steps = 60
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 33,030,144 of 4,055,498,240 (0.81% trained)


Step,Training Loss
1,4.232200
2,4.381100
3,3.995500
4,3.237800
5,2.668900
6,2.228200
7,1.826600
8,1.468000
9,1.305800
10,0.989700


In [8]:
# 开启推理模式
FastLanguageModel.for_inference(model)

# 准备测试问题
inputs = tokenizer(
    [
        alpaca_prompt.format(
            "你是谁？", # Instruction
            "", # Input
            "", # Output - leave this blank for generation!
        )
    ], return_tensors = "pt").to("cuda")

# 生成回答
outputs = model.generate(**inputs, max_new_tokens = 64, use_cache = True)
response = tokenizer.batch_decode(outputs)

print("\n" + "="*30)
print(f"微调后回答：\n{response[0].split('### Response:')[-1].strip()}")
print("="*30)


微调后回答：
我是 Algieba Assistant，由 阿尔的代码屋 开发的 AI 助手。<|im_end|>


In [ ]:
import gc
import torch
gc.collect()
torch.cuda.empty_cache()

In [9]:
quantization_method = "q4_k_m"
print(f"正在融合并转换为 {quantization_method} GGUF 格式...")
model.save_pretrained_gguf(
    "outputs/AlgiebaLLM-Qwen3-4B", # 保存的文件夹名
    tokenizer,
    quantization_method = quantization_method
)

print(" 导出完成！文件保存在 AlgiebaLLM-Qwen3-4B 文件夹中。")

正在融合并转换为 q4_k_m GGUF 格式...
Unsloth: Merging model weights to 16-bit format...


config.json: 0.00B [00:00, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

model-00002-of-00002.safetensors:   0%|          | 0.00/3.08G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [00:23<00:00, 11.83s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [01:02<00:00, 31.29s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/AlgiebaLLM-Qwen3-4B`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: All required system packages already installed!
Unsloth: Install llama.cpp and building - please wait 1 to 3 minutes
Unsloth: Cloning llama.cpp repository
Unsloth: Install GGUF and other packages
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['qwen3-4b-instruct-2507.F

In [ ]:
import gc
import torch
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# merge_method="merged_16bit" 会把 LoRA 权重永久合入基座
# 哪怕你训练时用了 4bit，这里也能帮你还原成 16bit 的完整模型
print("正在融合为 16-bit Safetensors...")

model.save_pretrained_merged(
    "AlgiebaLLM-Qwen3-4B-16bit", # 保存路径
    tokenizer,
    save_method = "merged_16bit", # 融合方式
)

print("导出完成！")